# 神經網路與自動微分互動教程

使用 `nn0.py` 中的純 Python autograd 引擎進行機器學習

## 課程大綱：
1. 導入和設置
2. 了解 Value 類和自動微分（Autograd）
3. 建構簡單神經網路模型
4. 使用 Adam 優化器實現訓練迴圈
5. 評估模型性能
6. 可視化損失和梯度流動

## 第 1 部分：導入和設置

In [ ]:
# 導入必要的模塊
import sys
sys.path.insert(0, '.')

from nn0 import Value, Adam, softmax
import random
import math
import matplotlib.pyplot as plt
import numpy as np

# 設置隨機種子以確保可重現性
random.seed(42)
np.random.seed(42)

print("✓ 所有模塊導入成功！")
print("✓ 隨機種子已設置")

# 設置繪圖風格
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 第 2 部分：了解 Value 類和自動微分

### 什麼是 Value？

`Value` 類是一個自動微分節點，能夠：
- 存儲數據（data）和梯度（grad）
- 記錄與其他 Value 的計算關係（_children, _local_grads）
- 執行反向傳播計算所有參數的梯度

### 工作流程：
1. **前向傳播（Forward Pass）**：執行計算，構建計算圖
2. **反向傳播（Backward Pass）**：沿著計算圖反向傳播梯度
3. **梯度累積**：每個參數累積來自所有計算路徑的梯度

In [ ]:
# 示例 1：簡單的計算圖
print("=" * 60)
print("示例 1：簡單計算圖")
print("=" * 60)

# 創建 Value 對象
a = Value(3.0)
b = Value(2.0)

print(f"\n初始化：")
print(f"a = {a}")
print(f"b = {b}")
print(f"a.grad = {a.grad}, b.grad = {b.grad}")

# 前向傳播：計算 c = a + b，d = c * b，z = d ** 2
c = a + b
d = c * b
z = d ** 2

print(f"\n預向傳播結果：")
print(f"c = a + b = {c}")
print(f"d = c * b = {d}")
print(f"z = d ** 2 = {z}")

# 反向傳播
z.backward()

print(f"\n反向傳播後的梯度：")
print(f"∂z/∂a = {a.grad}")
print(f"∂z/∂b = {b.grad}")
print(f"∂z/∂d = {d.grad}")

# 手動驗證
print(f"\n手動驗證：")
print(f"z = (a + b)² * b² = (3 + 2)² * 2² = 25 * 4 = 100 ✓")
print(f"∂z/∂a = 2 * (a+b) * b = 2 * 5 * 2 = 20 ✓")
print(f"∂z/∂b = 2 * (a+b) * b + (a+b)² * 2 = 2 * 5 * 2 + 25 * 2 = 20 + 50 = 70 ✓")

## 第 3 部分：建構簡單神經網路模型

In [ ]:
class MLP:
    """多層感知機 (Multi-Layer Perceptron)"""
    
    def __init__(self, layer_sizes):
        """
        初始化 MLP
        
        Args:
            layer_sizes: 列表 [input_size, hidden1, hidden2, ..., output_size]
        """
        self.layers = []
        
        for i in range(len(layer_sizes) - 1):
            in_size = layer_sizes[i]
            out_size = layer_sizes[i + 1]
            
            # 隨機初始化權重和偏置
            w = [[Value(random.gauss(0, 0.1)) for _ in range(in_size)] 
                 for _ in range(out_size)]
            b = [Value(0) for _ in range(out_size)]
            
            self.layers.append({'w': w, 'b': b})
    
    def __call__(self, x):
        """前向傳播"""
        for i, layer in enumerate(self.layers):
            # 線性變換：z = W @ x + b
            z = [sum(w_row[j] * x[j] for j in range(len(x))) + b 
                 for w_row, b in zip(layer['w'], layer['b'])]
            
            # 在隱藏層使用 ReLU 激活
            if i < len(self.layers) - 1:
                x = [Value(0) if z_i.data <= 0 else z_i for z_i in z]
            else:
                x = z  # 輸出層不使用激活
        
        return x
    
    def parameters(self):
        """取得所有參數"""
        params = []
        for layer in self.layers:
            for w_row in layer['w']:
                params.extend(w_row)
            params.extend(layer['b'])
        return params
    
    def zero_grad(self):
        """清除所有梯度"""
        for p in self.parameters():
            p.grad = 0

# 測試模型
print("\n建立 MLP 模型：輸入層 (2) → 隱藏層 (8) → 輸出層 (3)")
model = MLP([2, 8, 3])
print(f"✓ 模型已建立")
print(f"✓ 總參數數量: {len(model.parameters())}")

# 前向傳播測試
test_input = [Value(1.0), Value(2.0)]
output = model(test_input)
print(f"\n前向傳播測試：")
print(f"輸入: [1.0, 2.0]")
print(f"輸出層有 3 個單元:")
for i, val in enumerate(output):
    print(f"  輸出[{i}] = {val.data:.4f}")

## 第 4 部分：使用 Adam 優化器實現訓練迴圈

### XOR 問題：經典非線性分類問題

XOR (異或) 問題是神經網路中的經典問題：
- 輸入: 兩個二進制值 (0 或 1)
- 輸出: XOR(a, b)
- 表格:
  - XOR(0, 0) = 0
  - XOR(0, 1) = 1
  - XOR(1, 0) = 1
  - XOR(1, 1) = 0

這是一個**非線性可分**問題，證明單層感知機無法解決，需要至少一個隱藏層。

In [ ]:
# 訓練數據：XOR 問題
X_train = [
    [0, 0],  # XOR(0,0) = 0
    [0, 1],  # XOR(0,1) = 1
    [1, 0],  # XOR(1,0) = 1
    [1, 1],  # XOR(1,1) = 0
]
y_train = [0, 1, 1, 0]

# 建立模型
model = MLP([2, 16, 1])

# 建立 Adam 優化器
optimizer = Adam(model.parameters(), lr=0.1)

# 訓練迴圈
num_epochs = 200
losses_history = []

print("開始訓練 XOR 模型...")
print("=" * 60)

for epoch in range(num_epochs):
    model.zero_grad()
    
    total_loss = 0
    for x, y in zip(X_train, y_train):
        # 準備輸入
        x_val = [Value(xi) for xi in x]
        
        # 前向傳播
        logits = model(x_val)
        
        # MSE 損失函數
        pred = logits[0]
        loss = (pred - Value(y)) ** 2
        loss.backward()
        total_loss += loss.data
    
    # 計算平均損失
    avg_loss = total_loss / len(X_train)
    losses_history.append(avg_loss)
    
    # Adam 優化執行一步
    optimizer.step()
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1:3d}, 平均損失: {avg_loss:.6f}")

print("=" * 60)
print("訓練完成！\n")

# 測試預測結果
print("預測結果：")
print("-" * 40)
for x, y_true in zip(X_train, y_train):
    x_val = [Value(xi) for xi in x]
    pred = model(x_val)[0].data
    match = "✓" if abs(pred - y_true) < 0.2 else "✗"
    print(f"XOR({x[0]}, {x[1]}) = {pred:.4f}  (真值: {y_true})  {match}")
print("-" * 40)

## 第 5 部分：評估模型性能

In [ ]:
# 多類分類訓練
print("\n多類分類示例（3 個類別）")
print("=" * 60)

# 生成分類訓練數據
random.seed(42)
X_train_multi = []
y_train_multi = []

# 類別 0: 中心在 (0, 0)
for _ in range(15):
    x = random.gauss(0, 0.5)
    y = random.gauss(0, 0.5)
    X_train_multi.append([x, y])
    y_train_multi.append(0)

# 類別 1: 中心在 (3, 0)
for _ in range(15):
    x = random.gauss(3, 0.5)
    y = random.gauss(0, 0.5)
    X_train_multi.append([x, y])
    y_train_multi.append(1)

# 類別 2: 中心在 (1.5, 3)
for _ in range(15):
    x = random.gauss(1.5, 0.5)
    y = random.gauss(3, 0.5)
    X_train_multi.append([x, y])
    y_train_multi.append(2)

# 建立分類模型
model_multi = MLP([2, 8, 3])
optimizer_multi = Adam(model_multi.parameters(), lr=0.1)

# 訓練
num_epochs_multi = 150
losses_history_multi = []

for epoch in range(num_epochs_multi):
    model_multi.zero_grad()
    
    total_loss = 0
    for x, y_true in zip(X_train_multi, y_train_multi):
        x_val = [Value(xi) for xi in x]
        logits = model_multi(x_val)
        
        # Softmax + 交叉熵損失
        probs = softmax(logits)
        loss = -probs[y_true].log()
        loss.backward()
        total_loss += loss.data
    
    avg_loss = total_loss / len(X_train_multi)
    losses_history_multi.append(avg_loss)
    optimizer_multi.step()
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1:3d}, 平均損失: {avg_loss:.6f}")

# 評估準確度
print("\n評估分類準確度：")
print("-" * 40)
correct = 0
for x, y_true in zip(X_train_multi, y_train_multi):
    x_val = [Value(xi) for xi in x]
    logits = model_multi(x_val)
    probs = softmax(logits)
    pred = max(range(3), key=lambda i: probs[i].data)
    correct += (pred == y_true)

accuracy = correct / len(X_train_multi) * 100
print(f"訓練準確度: {accuracy:.1f}%")
print("-" * 40)

## 第 6 部分：可視化損失和梯度流動

In [ ]:
# 繪製訓練過程中的損失變化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 圖 1：XOR 損失曲線
axes[0].plot(losses_history, linewidth=2, color='#3498db')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('損失 (MSE)', fontsize=12)
axes[0].set_title('XOR 問題：訓練損失', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

# 圖 2：多類分類損失曲線
axes[1].plot(losses_history_multi, linewidth=2, color='#e74c3c')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('損失 (交叉熵)', fontsize=12)
axes[1].set_title('多類分類：訓練損失', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ 損失曲線已繪製")
print("\n觀察：")
print("- 損失隨著訓練逐漸降低")
print("- 曲線變得平緩表示收斂")
print("- 對數刻度更容易看到早期快速下降")

In [ ]:
# 梯度流動分析
print("\n梯度流動分析")
print("=" * 60)

# 計算單個樣點的損失和梯度
model.zero_grad()
x_test = [Value(1.0), Value(0.0)]
output = model(x_test)
loss = (output[0] - Value(1.0)) ** 2
loss.backward()

# 分析梯度大小
grad_magnitudes = []
for param in model.parameters():
    if param.grad != 0:
        grad_magnitudes.append(abs(param.grad))

if grad_magnitudes:
    print(f"\n梯度統計：")
    print(f"  最小梯度: {min(grad_magnitudes):.2e}")
    print(f"  最大梯度: {max(grad_magnitudes):.2e}")
    print(f"  平均梯度: {sum(grad_magnitudes) / len(grad_magnitudes):.2e}")
    print(f"  非零梯度數: {len(grad_magnitudes)} / {len(model.parameters())}")

# 可視化梯度分佈
if len(grad_magnitudes) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    
    # 繪製梯度直方圖
    ax.hist(grad_magnitudes, bins=20, color='#27ae60', alpha=0.7, edgecolor='black')
    ax.set_xlabel('梯度絕對值', fontsize=12)
    ax.set_ylabel('頻數', fontsize=12)
    ax.set_title('梯度分佈', fontsize=13, fontweight='bold')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ 梯度分佈已繪製")

## 進階：梯度檢查和學習率的影響

### 梯度檢查（Gradient Checking）

梯度檢查是一種驗證反向傳播實現正確性的技術。我們使用**數值梯度**（有限差分法）檢查**解析梯度**（反向傳播）。

如果兩者接近（通常誤差 < 1e-5），則實現正確。

In [ ]:
# 梯度檢查示例
print("\n梯度檢查示例")
print("=" * 60)

def numerical_gradient(func, x, eps=1e-5):
    """計算數值梯度（有限差分法）"""
    grad = Value(0)
    
    # f(x + eps)
    x_plus = Value(x.data + eps)
    f_plus = func(x_plus).data
    
    # f(x - eps)
    x_minus = Value(x.data - eps)
    f_minus = func(x_minus).data
    
    # 中心差分
    grad.data = (f_plus - f_minus) / (2 * eps)
    return grad

# 測試函數：z = x^3 + 2x^2
def test_func(x):
    return x**3 + Value(2) * x**2

# 測試點
x_test = Value(2.0)

# 反向傳播梯度
z = test_func(x_test)
z.backward()
analytical_grad = x_test.grad

# 數值梯度
numerical_grad = numerical_gradient(test_func, x_test).data

print(f"\n函數：z = x³ + 2x² 在 x = 2.0 處")
print(f"\n∂z/∂x:")
print(f"  解析梯度（反向傳播）: {analytical_grad:.8f}")
print(f"  數值梯度（有限差分）: {numerical_grad:.8f}")
print(f"  相對誤差: {abs(analytical_grad - numerical_grad) / (abs(analytical_grad) + 1e-8):.2e}")

if abs(analytical_grad - numerical_grad) < 1e-5:
    print(f"\n✓ 梯度檢查通過！反向傳播實現正確。")
else:
    print(f"\n✗ 梯度檢查失敗！可能存在實現錯誤。")

### 學習率的影響

學習率（Learning Rate）是優化過程中最重要的超參數之一：
- **太小**：訓練太慢，容易陷入局部最優
- **太大**：可能無法收斂，甚至損失發散
- **合適**：平衡收斂速度和穩定性

In [ ]:
# 比較不同學習率的影響
print("\n比較不同學習率的效果")
print("=" * 60)

learning_rates = [0.001, 0.01, 0.1, 0.5]
results = {}

for lr in learning_rates:
    # 建立新模型
    model_lr = MLP([2, 16, 1])
    optimizer_lr = Adam(model_lr.parameters(), lr=lr)
    
    losses = []
    for epoch in range(100):
        model_lr.zero_grad()
        
        total_loss = 0
        for x, y in zip(X_train, y_train):
            x_val = [Value(xi) for xi in x]
            logits = model_lr(x_val)
            pred = logits[0]
            loss = (pred - Value(y)) ** 2
            loss.backward()
            total_loss += loss.data
        
        avg_loss = total_loss / len(X_train)
        losses.append(avg_loss)
        optimizer_lr.step()
    
    results[lr] = losses
    print(f"LR = {lr:5.3f}: 初始損失 = {losses[0]:.6f}, 最終損失 = {losses[-1]:.6f}")

# 繪製對比
fig, ax = plt.subplots(figsize=(12, 6))

for lr in learning_rates:
    ax.plot(results[lr], linewidth=2, label=f'LR = {lr}', marker='o', markersize=4, markevery=10)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('損失 (MSE)', fontsize=12)
ax.set_title('不同學習率對 XOR 訓練的影響', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ 學習率對比已繪製")

## 總結和要點

### 核心概念回顧：

1. **Value 類**：自動微分節點
   - 記錄數據和計算關係
   - 支援轉發傳播構建計算圖
   - 反向傳播計算梯度

2. **神經網路**：多層感知機 (MLP)
   - 線性層 + 非線性激活
   - ReLU：隱藏層
   - 無激活：輸出層（供損失函數使用）

3. **訓練過程**：
   - 前向傳播：計算預測
   - 損失函數：量化誤差（MSE、交叉熵）
   - 反向傳播：計算梯度
   - 優化器：更新參數（Adam）

4. **超參數調優**：
   - 學習率：控制參數更新幅度
   - 網路架構：層數、每層寬度
   - 訓練輪次：平衡準確度和過擬合

### 實踐技巧：

✓ **梯度檢查**：驗證反向傳播實現  
✓ **監控損失**：及早發現訓練問題  
✓ **調整學習率**：改進收斂速度  
✓ **評估指標**：選擇合適的評價標準（準確度、F1 等）

### 進一步探索：

- 添加正則化（L1/L2）防止過擬合
- 實現批量梯度下降 (Mini-batch GD)
- 嘗試不同的網路架構
- 在實際數據集上測試（MNIST、CIFAR-10）

## 互動練習題

### 練習 1：修改網路架構
試試改變 MLP 的層數或每層的寬度。觀察如何影響訓練速度和最終性能。

### 練習 2：函數擬合
訓練一個簡單的 MLP 來擬合 `sin(x)` 或 `cos(x)` 函數。

### 練習 3：自訂損失函數
實現其他損失函數（如 L1 損失、Huber 損失），比較它們的性能。

### 練習 4：梯度可視化
繪製網路各層的梯度分佈，觀察梯度消失/爆炸的現象。

祝您學習愉快！🚀